In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

assert torch.cuda.is_available(), (
    "GPU unavailable. Select a GPU runtime and reconnect."
)

device = torch.cuda.get_device_properties(0)

print("GPU:", device.name)
print(f"Total GPU memory: {device.total_memory / 1024**3:.2f} GiB")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
PyTorch CUDA version: 13.0
GPU: Tesla T4
Total GPU memory: 14.56 GiB


In [2]:
!nvidia-smi

Wed Oct  7 11:18:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
x = torch.randn(1024, 1024, device="cuda")
y = x @ x.T

torch.cuda.synchronize()

print("Result device:", y.device)
print("Result shape:", tuple(y.shape))
print("GPU calculation successful.")

del x, y
torch.cuda.empty_cache()

Result device: cuda:0
Result shape: (1024, 1024)
GPU calculation successful.


In [5]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory GiB:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))

Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
Memory GiB: 14.56


In [6]:
import torch

x = torch.randn(1024, 1024, device="cuda")
y = x @ x.T
torch.cuda.synchronize()

print("Device:", y.device)
print("Shape:", tuple(y.shape))
print("All values finite:", torch.isfinite(y).all().item())

del x, y
torch.cuda.empty_cache()

Device: cuda:0
Shape: (1024, 1024)
All values finite: True


In [7]:
%pip install -q -U transformers accelerate safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 112.1 MB/s eta 0:00:00


In [8]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Connect to a GPU runtime first."

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("Transformers:", transformers.__version__)
print("Loading:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.eval()

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB",
)

Transformers: 5.19.0
Loading: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model device: cuda:0
Model dtype: torch.float16
Allocated GPU memory: 2.88 GiB


In [9]:
context = """
[doc1] ContextFit Support provides assistance Monday to Friday,
from 9 AM to 6 PM IST. Customers can request a refund within
14 days of purchase by contacting support.
"""

question = "Within how many days can a customer request a refund?"

messages = [
    {
        "role": "system",
        "content": (
            "Answer using only the supplied context. "
            "Cite the supporting document ID, such as [doc1]. "
            "If the answer is missing, say you do not know. "
            "Keep the answer brief."
        ),
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {question}",
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Answer:", answer)
print("Input tokens:", inputs["input_ids"].shape[1])
print("Generated tokens:", len(new_tokens))

Answer: A customer can request a refund within 14 days of purchase. [doc1]
Input tokens: 104
Generated tokens: 19


In [11]:
import time
import pandas as pd

def measure_generation(inputs, max_new_tokens=80):
    torch.cuda.synchronize()
    baseline = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    peak = torch.cuda.max_memory_allocated()

    input_count = inputs["input_ids"].shape[1]
    generated = output[0, input_count:]
    output_count = generated.numel()

    result = {
        "input_tokens": input_count,
        "output_tokens": output_count,
        "generation_seconds": elapsed,
        "output_tokens_per_second": output_count / elapsed,
        "baseline_allocated_gib": baseline / (1024**3),
        "peak_allocated_gib": peak / (1024**3),
        "extra_peak_allocated_mib": (peak - baseline) / (1024**2),
        "answer": tokenizer.decode(generated, skip_special_tokens=True),
    }

    return result

In [12]:
for _ in range(2):
    _ = measure_generation(inputs)

records = []

for run in range(1, 6):
    record = measure_generation(inputs)
    record["run"] = run
    records.append(record)

df = pd.DataFrame(records)

print(df[[
    "run",
    "input_tokens",
    "output_tokens",
    "generation_seconds",
    "peak_allocated_gib",
]].round(3).to_string(index=False))

print(
    "\nMedian generation time:",
    round(df["generation_seconds"].median(), 3),
    "seconds",
)

 run  input_tokens  output_tokens  generation_seconds  peak_allocated_gib
   1           104             19               1.087               2.894
   2           104             19               1.167               2.894
   3           104             19               0.705               2.894
   4           104             19               0.687               2.894
   5           104             19               0.678               2.894

Median generation time: 0.705 seconds


In [13]:
def prepare_inputs(question, context):
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied context. "
                "Cite the supporting document IDs, such as [doc1]. "
                "If the answer is missing, say you do not know. "
                "Keep the answer brief."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}",
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")


def run_record(record):
    prepared = prepare_inputs(
        question=record["question"],
        context=record["context"],
    )

    result = measure_generation(prepared)

    result.update({
        "question_id": record["question_id"],
        "policy": record["policy"],
        "source_ids": record["source_ids"],
    })

    return result

In [14]:
sample = {
    "question_id": "smoke_002",
    "policy": "manual_context",
    "question": "What is the annual subscription price?",
    "context": (
        "[doc1] Customers can request a refund within "
        "14 days of purchase by contacting support."
    ),
    "source_ids": ["doc1"],
}

result = run_record(sample)

print("Answer:", result["answer"])
print("Input tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])
print("Generation seconds:", round(result["generation_seconds"], 3))

Answer: I do not know. The provided context does not contain information about an annual subscription price.
Input tokens: 81
Output tokens: 19
Generation seconds: 1.049


In [15]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

RESULTS_DIR = Path("/content/drive/MyDrive/ContextFit/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Results folder:", RESULTS_DIR)

Mounted at /content/drive
Results folder: /content/drive/MyDrive/ContextFit/results


In [16]:
import json
import sys
import subprocess
from datetime import datetime, timezone
import transformers

SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
SESSION_DIR = RESULTS_DIR / SESSION_ID
SESSION_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "session_id": SESSION_ID,
    "python": sys.version,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "dtype": str(next(model.parameters()).dtype),
    "max_new_tokens": 80,
    "do_sample": False,
    "timing_scope": "model.generate only; inputs already on GPU",
}

(SESSION_DIR / "environment.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

packages = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)

(SESSION_DIR / "requirements-colab.txt").write_text(
    packages, encoding="utf-8"
)

df.to_csv(SESSION_DIR / "first_inference.csv", index=False)

(SESSION_DIR / "unanswerable_check.json").write_text(
    json.dumps(
        {"request": sample, "result": result},
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved session:", SESSION_DIR)

Saved session: /content/drive/MyDrive/ContextFit/results/20261007T112948_818950Z


In [17]:
import traceback

RESULTS_FILE = SESSION_DIR / "requests.jsonl"

def run_and_save(record):
    entry = {
        "session_id": SESSION_ID,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "request": record,
    }

    try:
        entry["result"] = run_record(record)
        entry["status"] = "ok"
    except Exception as exc:
        entry["status"] = "error"
        entry["error_type"] = type(exc).__name__
        entry["error"] = str(exc)
        entry["traceback"] = traceback.format_exc()

    with RESULTS_FILE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(entry, ensure_ascii=False) + "\n")

    return entry

In [18]:
saved = run_and_save(sample)

print("Status:", saved["status"])
print("Saved to:", RESULTS_FILE)

if saved["status"] == "ok":
    print("Answer:", saved["result"]["answer"])
else:
    print("Error:", saved["error"])

Status: ok
Saved to: /content/drive/MyDrive/ContextFit/results/20261007T112948_818950Z/requests.jsonl
Answer: I do not know. The provided context does not contain information about an annual subscription price.


In [19]:
smoke_records = [
    {
        "question_id": "smoke_001",
        "policy": "manual_context",
        "question": "Within how many days can I request a refund?",
        "context": (
            "[doc1] Customers can request a refund within "
            "14 days of purchase."
        ),
        "source_ids": ["doc1"],
    },
    {
        "question_id": "smoke_002",
        "policy": "manual_context",
        "question": "What is the annual subscription price?",
        "context": (
            "[doc1] Customers can request a refund within "
            "14 days of purchase."
        ),
        "source_ids": ["doc1"],
    },
    {
        "question_id": "smoke_003",
        "policy": "manual_context",
        "question": "On which days is support available?",
        "context": (
            "[doc2] Support is available Monday to Friday, "
            "from 9 AM to 6 PM IST."
        ),
        "source_ids": ["doc2"],
    },
]

In [20]:
import hashlib
import random

BATCH_FILE = SESSION_DIR / "smoke_batch.jsonl"

def make_job_id(record, repeat):
    payload = {
        "request": record,
        "repeat": repeat,
        "configuration": metadata,
        "runner_version": "smoke_v1",
    }
    encoded = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def run_batch(records, repeats=3):
    completed = set()

    if BATCH_FILE.exists():
        for line in BATCH_FILE.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            entry = json.loads(line)
            if entry["status"] == "ok":
                completed.add(entry["job_id"])

    jobs = [
        (record, repeat)
        for repeat in range(1, repeats + 1)
        for record in records
    ]

    random.Random(42).shuffle(jobs)

    for record, repeat in jobs:
        job_id = make_job_id(record, repeat)
        label = f'{record["question_id"]} | repeat {repeat}'

        if job_id in completed:
            print("SKIP:", label)
            continue

        entry = {
            "job_id": job_id,
            "session_id": SESSION_ID,
            "repeat": repeat,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "request": record,
        }

        try:
            entry["result"] = run_record(record)
            entry["status"] = "ok"
        except Exception as exc:
            entry["status"] = "error"
            entry["error_type"] = type(exc).__name__
            entry["error"] = str(exc)

        with BATCH_FILE.open("a", encoding="utf-8") as file:
            file.write(json.dumps(entry, ensure_ascii=False) + "\n")

        if entry["status"] == "ok":
            completed.add(job_id)

        print(entry["status"].upper() + ":", label)

In [21]:
run_batch(smoke_records, repeats=3)

OK: smoke_001 | repeat 2
OK: smoke_001 | repeat 3
OK: smoke_002 | repeat 3
OK: smoke_002 | repeat 2
OK: smoke_003 | repeat 3
OK: smoke_003 | repeat 1
OK: smoke_003 | repeat 2
OK: smoke_001 | repeat 1
OK: smoke_002 | repeat 1


In [22]:
run_batch(smoke_records, repeats=3)

SKIP: smoke_001 | repeat 2
SKIP: smoke_001 | repeat 3
SKIP: smoke_002 | repeat 3
SKIP: smoke_002 | repeat 2
SKIP: smoke_003 | repeat 3
SKIP: smoke_003 | repeat 1
SKIP: smoke_003 | repeat 2
SKIP: smoke_001 | repeat 1
SKIP: smoke_002 | repeat 1


In [23]:
import ast
from pathlib import Path

MODULE_DIR = Path(
    "/content/drive/MyDrive/ContextFit/src/contextfit"
)
MODULE_DIR.mkdir(parents=True, exist_ok=True)

wanted = {
    "measure_generation",
    "prepare_inputs",
    "run_record",
}
definitions = {}

for cell in get_ipython().history_manager.input_hist_raw:
    try:
        tree = ast.parse(cell)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in wanted:
            definitions[node.name] = ast.get_source_segment(cell, node)

missing = wanted - definitions.keys()
if missing:
    raise RuntimeError(f"Missing function definitions: {sorted(missing)}")

header = '''import time
import torch

model = None
tokenizer = None

def configure(loaded_model, loaded_tokenizer):
    global model, tokenizer
    model = loaded_model
    tokenizer = loaded_tokenizer

'''

source = header + "\n\n".join(
    definitions[name]
    for name in (
        "measure_generation",
        "prepare_inputs",
        "run_record",
    )
) + "\n"

compile(source, "inference.py", "exec")

module_path = MODULE_DIR / "inference.py"
module_path.write_text(source, encoding="utf-8")
(MODULE_DIR / "__init__.py").touch()

print("Saved:", module_path)

Saved: /content/drive/MyDrive/ContextFit/src/contextfit/inference.py


In [24]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "contextfit_inference", module_path
)
inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference)

inference.configure(model, tokenizer)

module_result = inference.run_record(smoke_records[0])

print("Answer:", module_result["answer"])
print(
    "Generation seconds:",
    round(module_result["generation_seconds"], 3),
)

Answer: Within 14 days of purchase. [doc1]
Generation seconds: 0.723


In [25]:
from google.colab import files

files.download(str(module_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>